# µPMU v2 — Caracterización y resultados finales
**Alejandro Ariza y Laura Sofía Rico — Taller de Potencia 2026-20**

Este cuaderno lee **todos** los archivos que guarda `capturar_upmu.py` en la carpeta `datos` y calcula:

1. **Calibración**: `K_V`, `K_I` y el desfase entre los sensores (con la carga resistiva y el multímetro).
2. **Exactitud**: errores de magnitud, error de ángulo V–I y **TVE**.
3. **Frecuencia y ROCOF**: FE y RFE.
4. **Sincronía**: coherencia de la fase con el PPS y error del reloj del MCU.
5. **Latencia** del reporte de fasores calculado en el MCU.
6. **Algoritmo**: cuánto se degrada fuera de 60 Hz (simulación) y cuánto difiere de un ajuste sinusoidal óptimo (datos reales).
7. **Tabla final de criterios de aceptación** (IEEE C37.118.1 y criterios propios), guardada en `resultados/`.

**Cómo usarlo:** pon tus archivos en `datos/`, completa los parámetros de la celda siguiente y ejecuta todo (*Run All*).
Si `datos/` está vacía, el cuaderno genera datos **SIMULADOS** solo para mostrar que funciona: todos los títulos lo dicen. **No son mediciones.**

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.2), "axes.grid": True, "grid.alpha": 0.3})

# ---------------- carpetas ----------------
CARPETA_DATOS = "datos"            # donde capturar_upmu.py guarda los archivos
CARPETA_RESULTADOS = "resultados"  # tablas y figuras

# ---------------- parámetros del montaje ----------------
P = dict(
    F0=60.0,
    CORRECCION_RETARDO_US=0.0,   # retardo real PPS->muestra menos el del firmware (de la medida con osciloscopio en PB3)
    INCERT_RETARDO_US=2.0,       # incertidumbre de ese retardo: 2 us si no se midió, 0,3 us si se midió con osciloscopio
    FASE_I_SUPUESTA_DEG=1.14,    # adelanto teórico del canal de corriente (red Rg-Cg del LM358)
    FASE_I_MEDIDA_OSC_DEG=None,  # desfase del canal de corriente medido con osciloscopio (grados, + = adelanta)
    FASE_V_MEDIDA_OSC_DEG=None,  # desfase del canal de tensión medido con osciloscopio (usa solo uno de los dos)
    INCERT_FASE_ABS_DEG=2.0,     # incertidumbre del ángulo absoluto de los sensores:
                                 #   2° sin medir (el SCT-013 no trae dato de fase); ~0,2° si se midió con osciloscopio
    EXACT_MULT_V_PCT=1.0,        # exactitud del multímetro en V AC (manual del multímetro)
    EXACT_MULT_I_PCT=1.5,        # exactitud del multímetro en A AC
    K_V_FIRMWARE=0.00108,        # constantes que tenía el firmware antes de calibrar (para "error encontrado")
    K_I_FIRMWARE=0.1965,
    REPORTES_POR_S=1.0,          # tasa de reporte del modo m
)

# ---------------- criterios de aceptación ----------------
CRIT = dict(
    TVE_PCT=1.0,          # IEEE C37.118.1, estado estable (P y M)
    FE_HZ=0.005,          # IEEE C37.118.1, estado estable
    RFE_P_HZS=0.4,        # IEEE C37.118.1a-2014, clase P (rango de frecuencia)
    RFE_M_HZS=0.1,        # IEEE C37.118.1a-2014, clase M
    LATENCIA_FACTOR_P=2,  # latencia <= 2/Fs (clase P)
    ERR_V_PCT=1.0,        # propio: error de magnitud de tensión
    ERR_I_PCT=3.0,        # propio: error de magnitud de corriente (no linealidad del SCT-013: ±3 %)
    ERR_ANG_DEG=0.573,    # propio: error de ángulo V–I equivalente a TVE = 1 %
    SINC_US=26.5,         # error de tiempo equivalente a TVE = 1 % a 60 Hz
)

## Funciones (no hace falta modificarlas)
La función `dft_upmu` es **el mismo algoritmo que corre en el MCU** (se comprobó que da los mismos números). `ajuste_seno` es un ajuste de 4 parámetros (IEEE 1057) que sirve como referencia fuera de línea.

In [ ]:
import numpy as np

DOSPI = 2 * np.pi


def envolver(a):
    """Angulo en rad a (-pi, pi]."""
    return (np.asarray(a) + np.pi) % DOSPI - np.pi


def envolver_g(a):
    """Angulo en grados a (-180, 180]. Conserva el tipo (numero, arreglo o serie de pandas)."""
    return (a + 180.0) % 360.0 - 180.0


def tiempos(n, fs_real, retardo_s):
    """Instante de cada muestra respecto al PPS: t_k = (k+1)/fs + retardo."""
    return (np.arange(n) + 1) / fs_real + retardo_s


def dft_upmu(v, i, ppc, fs_real, retardo_s, lsb):
    """Mismo algoritmo que el firmware (dspCalcular): DFT de 16 ciclos en 2 mitades.
    Devuelve rms en V del pin, angulos en rad referidos al PPS y frecuencia en Hz."""
    v = np.asarray(v, float)
    i = np.asarray(i, float)
    n = len(v)
    xv, xi = v - v.mean(), i - i.mean()
    e = np.exp(-1j * DOSPI * (np.arange(n) % ppc) / ppc)
    h = n // 2
    Xv = (np.sum(xv[:h] * e[:h]), np.sum(xv[h:] * e[h:]))
    Xi = (np.sum(xi[:h] * e[:h]), np.sum(xi[h:] * e[h:]))
    esc = np.sqrt(2) / n * lsb
    V = (Xv[0] + Xv[1]) * esc
    I = (Xi[0] + Xi[1]) * esc
    fbin = fs_real / ppc
    usar_v = abs(V) > 20 * lsb or abs(V) >= abs(I)
    a1, a2 = (np.angle(Xv[0]), np.angle(Xv[1])) if usar_v else (np.angle(Xi[0]), np.angle(Xi[1]))
    dt = (n / 2) / fs_real
    df = envolver(a2 - a1) / (DOSPI * dt)
    tmed = ((n - 1) / 2 + 1) / fs_real + retardo_s
    corr = DOSPI / ppc + DOSPI * fbin * retardo_s + DOSPI * df * tmed
    D = 1.0                                  # atenuacion de la ventana fuera de 60 Hz (Dirichlet)
    xd = np.pi * df / fs_real
    if abs(xd) > 1e-7:
        D = np.sin(xd * n) / (n * np.sin(xd))
        D = D if D > 0.5 else 1.0
    return dict(V=abs(V) / D, angV=float(envolver(np.angle(V) - corr)),
                I=abs(I) / D, angI=float(envolver(np.angle(I) - corr)),
                f=fbin + df, f_ok=bool((abs(V) if usar_v else abs(I)) > 20 * lsb),
                mediaV=v.mean(), mediaI=i.mean())


def ajuste_seno(x, t, f_ini, ajustar_f=True, iter_max=8):
    """Ajuste sinusoidal por minimos cuadrados (IEEE 1057).
    x(t) = A cos(wt) + B sin(wt) + C.  Con ajustar_f=True tambien estima w (4 parametros).
    Devuelve rms, angulo (rad) en t = 0, frecuencia (Hz), C y rms del residuo."""
    x = np.asarray(x, float)
    w = DOSPI * f_ini
    for _ in range(iter_max if ajustar_f else 1):
        c, s = np.cos(w * t), np.sin(w * t)
        if ajustar_f:
            A0, B0 = _tres(x, c, s)[:2]
            M = np.column_stack([c, s, np.ones_like(t), t * (-A0 * s + B0 * c)])
            p, *_ = np.linalg.lstsq(M, x, rcond=None)
            dw = p[3]
            w += dw
            if abs(dw) < 1e-9 * w:
                break
    c, s = np.cos(w * t), np.sin(w * t)
    A, B, C = _tres(x, c, s)
    res = x - (A * c + B * s + C)
    R = np.hypot(A, B)
    return dict(rms=R / np.sqrt(2), ang=float(np.arctan2(-B, A)), f=w / DOSPI, C=C,
                res_rms=float(np.sqrt(np.mean(res ** 2))))


def _tres(x, c, s):
    M = np.column_stack([c, s, np.ones_like(c)])
    p, *_ = np.linalg.lstsq(M, x, rcond=None)
    return p


def armonicos(x, ppc, n_arm=13):
    """Amplitud relativa de los armonicos (DFT de ciclos enteros). Devuelve (THD %, lista % del fundamental)."""
    x = np.asarray(x, float) - np.mean(x)
    n = len(x) - len(x) % ppc
    X = np.abs(np.fft.rfft(x[:n])) / n * 2
    ciclos = n // ppc
    fund = X[ciclos]
    arm = [X[h * ciclos] / fund * 100 for h in range(2, n_arm + 1) if h * ciclos < len(X)]
    return float(np.sqrt(np.sum(np.square(arm)))), arm


def tve(fasor_med, fasor_ref):
    """TVE en % entre fasores complejos."""
    return float(abs(fasor_med - fasor_ref) / abs(fasor_ref) * 100)

In [ ]:
import glob
import os

import numpy as np
import pandas as pd



# ============================================================ lectura
def _num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


def leer_archivo(ruta):
    meta, fw, bloques, reportes, lat, pps = {}, {}, [], [], {}, []
    cols_rep = None
    cur, estado, t_pc = None, None, np.nan
    with open(ruta, encoding="utf-8", errors="replace") as fh:
        for lin in fh:
            lin = lin.strip()
            if not lin:
                continue
            if lin.startswith("#meta_"):
                k, _, v = lin[6:].partition("=")
                meta[k] = v
                continue
            if lin.startswith("#UPMU"):
                cur, estado = dict(cab={}, v=[], i=[]), "cab"
                continue
            if estado in ("cab", "datos"):
                if lin == "#FIN":
                    if cur["v"]:
                        bloques.append(cur)
                    estado = None
                    continue
                if lin.startswith("#"):
                    k, _, v = lin[1:].partition("=")
                    cur["cab"][k] = v
                    continue
                if lin.startswith("adcV") or lin.startswith("n,"):
                    estado = "datos"
                    continue
                p = lin.split(",")
                try:
                    if len(p) == 2:
                        cur["v"].append(int(p[0])); cur["i"].append(int(p[1]))
                    elif len(p) == 3:
                        cur["v"].append(int(p[1])); cur["i"].append(int(p[2]))
                except ValueError:
                    pass
                continue
            if lin.startswith("$H"):
                cols_rep = lin.split(",")[1:]
            elif lin.startswith("$F") and cols_rep:
                vals = lin.split(",")[1:]
                reportes.append({c: _num(v) for c, v in zip(cols_rep, vals)})
            elif lin.startswith("$L"):
                p = lin.split(",")
                lat[int(p[1])] = int(p[2])
            elif lin.startswith("$P"):
                pps.append(int(lin.split(",")[1]))
            elif lin.startswith("#") and "=" in lin and not lin.startswith("#meta_"):
                k, _, v = lin[1:].partition("=")
                fw[k] = v
    rep = pd.DataFrame(reportes)
    if len(rep):
        rep["lat_total_us"] = rep["seg"].map(lambda s: lat.get(int(s), np.nan))
    return dict(ruta=ruta, nombre=os.path.basename(ruta), meta=meta, fw=fw, bloques=bloques,
                reportes=rep, pps=np.array(pps, float))


def leer_carpeta(carpeta):
    rutas = sorted(glob.glob(os.path.join(carpeta, "*.csv")))
    return [leer_archivo(r) for r in rutas]


# ============================================================ analisis por bloque
def periodo_global(archivos):
    """Periodo del PPS medido en cualquier archivo (mediana). Sirve para corregir bloques sin PPS."""
    pers = [_num(b["cab"].get("pps_periodo_us", 0)) for a in archivos for b in a["bloques"]]
    pers = [p for p in pers if np.isfinite(p) and p > 0]
    pers += [p for a in archivos for p in a["pps"] if 980000 < p < 1020000]
    return float(np.median(pers)) if pers else np.nan


def analizar_bloques(archivos, P):
    filas = []
    per_g = periodo_global(archivos)
    for a in archivos:
        m = a["meta"]
        for nb, b in enumerate(a["bloques"]):
            c = b["cab"]
            fs = _num(c.get("fs", 4800))
            ppc = int(_num(c.get("ppc", fs / P["F0"])))
            per = _num(c.get("pps_periodo_us", 0))
            origen = "PPS del bloque"
            if not (np.isfinite(per) and per > 0):
                per, origen = (per_g, "PPS de otra prueba") if np.isfinite(per_g) else (np.nan, "nominal (sin corregir)")
            fs_r = fs * per / 1e6 if np.isfinite(per) else fs
            ret = (_num(c.get("retardo_us", 0)) + P["CORRECCION_RETARDO_US"]) * 1e-6
            vref = _num(c.get("vref", 3.3))
            lsb = vref / _num(c.get("adcmax", 4095))
            v, i = np.array(b["v"]), np.array(b["i"])
            n = len(v)
            if n < 2 * ppc:
                continue
            d = dft_upmu(v, i, ppc, fs_r, ret, lsb)
            t = tiempos(n, fs_r, ret)
            fv = ajuste_seno(v * lsb, t, d["f"])
            fi = ajuste_seno(i * lsb, t, fv["f"], ajustar_f=False)
            filas.append(dict(
                archivo=a["nombre"], prueba=m.get("prueba", ""), rol=m.get("rol", "otra"), modo=m.get("modo", ""),
                bloque=nb, pps=int(_num(c.get("pps", 0)) or 0), seg=_num(c.get("seg", np.nan)),
                t_pc=_num(c.get("t_pc", np.nan)), fs_real=fs_r, fs_origen=origen, ppm=(per - 1e6) if np.isfinite(per) else np.nan,
                ref_V=_num(m.get("ref_V")), ref_I=_num(m.get("ref_I")), ref_ang=_num(m.get("ref_ang")),
                ref_f=_num(m.get("ref_f")),
                mediaV=d["mediaV"], mediaI=d["mediaI"],
                satV=bool(v.min() <= 5 or v.max() >= 4090), satI=bool(i.min() <= 5 or i.max() >= 4090),
                Vpin=d["V"], Ipin=d["I"], angV=np.degrees(d["angV"]), angI=np.degrees(d["angI"]), f=d["f"],
                f_ok=d["f_ok"],
                Vpin_aj=fv["rms"], Ipin_aj=fi["rms"], angV_aj=np.degrees(fv["ang"]), angI_aj=np.degrees(fi["ang"]),
                f_aj=fv["f"], resV_mV=fv["res_rms"] * 1e3, resI_mV=fi["res_rms"] * 1e3,
                thdV=armonicos(v, ppc)[0], thdI=armonicos(i, ppc)[0] if d["I"] > 20 * lsb else np.nan,
                lsb=lsb))
    return pd.DataFrame(filas)


# ============================================================ calibracion
def media_circular(g):
    g = np.radians(np.asarray(g, float))
    return float(np.degrees(np.angle(np.mean(np.exp(1j * g)))))


def calibrar(df, P):
    cal = df[df.rol == "calibracion"]
    out = dict(n_bloques=len(cal), archivos=sorted(cal.archivo.unique()))
    sel = cal[np.isfinite(cal.ref_V) & (cal.Vpin > 20 * cal.lsb)]
    rv = sel.Vpin / sel.ref_V
    out["K_V"], out["K_V_sd"], out["n_V"] = rv.mean(), rv.std(), len(sel)
    sel = cal[np.isfinite(cal.ref_I) & (cal.Ipin > 20 * cal.lsb)]
    ri = sel.Ipin / sel.ref_I
    out["K_I"], out["K_I_sd"], out["n_I"] = ri.mean(), ri.std(), len(sel)
    sel = cal[np.isfinite(cal.ref_ang) & (cal.Ipin > 20 * cal.lsb) & (cal.Vpin > 20 * cal.lsb)]
    dif = envolver_g(sel.angV - sel.angI - sel.ref_ang)
    out["desfase_VI"] = media_circular(dif) if len(sel) else np.nan
    out["desfase_VI_sd"] = float(np.std(envolver_g(dif - out["desfase_VI"]))) if len(sel) else np.nan
    out["n_ang"] = len(sel)
    # reparto del desfase: el canal de corriente se supone con el desfase teorico de Cg
    out["CAL_ANG_I"] = P["FASE_I_SUPUESTA_DEG"]
    out["CAL_ANG_V"] = out["desfase_VI"] + P["FASE_I_SUPUESTA_DEG"] if np.isfinite(out["desfase_VI"]) else np.nan
    out["fase_origen"] = "supuesta (red Rg-Cg)"
    if P.get("FASE_I_MEDIDA_OSC_DEG") is not None:          # medida con osciloscopio: manda la medida
        out["CAL_ANG_I"] = P["FASE_I_MEDIDA_OSC_DEG"]
        out["CAL_ANG_V"] = out["desfase_VI"] + out["CAL_ANG_I"]
        out["fase_origen"] = "canal I medido con osciloscopio"
    if P.get("FASE_V_MEDIDA_OSC_DEG") is not None:
        out["CAL_ANG_V"] = P["FASE_V_MEDIDA_OSC_DEG"]
        out["CAL_ANG_I"] = out["CAL_ANG_V"] - out["desfase_VI"]
        out["fase_origen"] = "canal V medido con osciloscopio"
    return out


def aplicar_cal(df, cal):
    d = df.copy()
    d["V"] = d.Vpin / cal["K_V"]
    d["I"] = d.Ipin / cal["K_I"]
    d["angV_c"] = envolver_g(d.angV - cal["CAL_ANG_V"])
    d["angI_c"] = envolver_g(d.angI - cal["CAL_ANG_I"])
    d["angVI"] = envolver_g(d.angV_c - d.angI_c)
    d["P"] = d.V * d.I * np.cos(np.radians(d.angVI))
    d["Q"] = d.V * d.I * np.sin(np.radians(d.angVI))
    d["FP"] = np.cos(np.radians(d.angVI))
    return d


# ============================================================ exactitud
def exactitud(d, P):
    val = d[d.rol == "validacion"]
    aviso = ""
    if len(val) == 0:
        val = d[d.rol == "calibracion"]
        aviso = ("No hay pruebas con rol 'validacion': se evalua con los mismos datos de calibracion. "
                 "El resultado es optimista; repite con otra carga o en otro momento.")
    r = dict(aviso=aviso, n=len(val), archivos=sorted(val.archivo.unique()))
    sv = val[np.isfinite(val.ref_V)]
    eV = (sv.V / sv.ref_V - 1) * 100
    si = val[np.isfinite(val.ref_I) & (val.Ipin > 20 * val.lsb)]
    eI = (si.I / si.ref_I - 1) * 100
    sa = si[np.isfinite(si.ref_ang)]
    ea = envolver_g(sa.angVI - sa.ref_ang)
    tveI = np.abs((sa.I / sa.ref_I) * np.exp(1j * np.radians(ea)) - 1) * 100
    # presupuesto angular absoluto (respecto al UTC) para la tension
    u_t = P["INCERT_RETARDO_US"] * 1e-6 * 360 * P["F0"]
    u_ang = np.hypot(P["INCERT_FASE_ABS_DEG"], u_t)
    r.update(eV=eV, eI=eI, ea=ea, tveI=tveI, u_ang_abs=u_ang)
    if len(eV):
        tveV = np.sqrt(eV.values ** 2 + (np.radians(u_ang) * 100) ** 2)
        r["tveV"] = pd.Series(tveV, index=eV.index)
    else:
        r["tveV"] = pd.Series(dtype=float)
    return r


# ============================================================ frecuencia, ROCOF y sincronia
def pares_consecutivos(d):
    """ROCOF entre bloques consecutivos del mismo archivo (ambos con PPS)."""
    filas = []
    for arch, g in d[(d.pps == 1) & d.f_ok].groupby("archivo"):
        g = g.sort_values("seg")
        for (_, a), (_, b) in zip(g.iloc[:-1].iterrows(), g.iloc[1:].iterrows()):
            dt = b.seg - a.seg
            if 0 < dt <= 10:
                filas.append(dict(archivo=arch, seg=b.seg, dt=dt, rocof=(b.f - a.f) / dt,
                                  rocof_aj=(b.f_aj - a.f_aj) / dt))
    return pd.DataFrame(filas)


def coherencia_sincronia(d, medio=3):
    """Prueba de sincronia: la fase de la tension en cada PPS debe seguir una curva suave
    (la red cambia de frecuencia despacio). Para cada bloque se predice su fase con una parabola
    ajustada a los vecinos (sin el) y se mide la diferencia, pasada a microsegundos."""
    filas = []
    for arch, g in d[(d.pps == 1) & d.f_ok].groupby("archivo"):
        g = g.sort_values("seg")
        if len(g) < 2 * medio + 1:
            continue
        t = g.seg.values.astype(float)
        th = g.angV_aj.values.astype(float)
        f = g.f_aj.values
        tu = np.empty_like(th)
        tu[0] = th[0]
        for k in range(1, len(th)):                       # desenvolver con el avance esperado
            pred = tu[k - 1] + 360 * ((f[k - 1] + f[k]) / 2 - 60.0) * (t[k] - t[k - 1])
            tu[k] = pred + envolver_g(th[k] - pred)
        for k in range(len(th)):
            idx = [j for j in range(max(0, k - medio), min(len(th), k + medio + 1)) if j != k]
            if len(idx) < 4:
                continue
            c = np.polyfit(t[idx] - t[k], tu[idx], 2)
            res = tu[k] - c[-1]
            filas.append(dict(archivo=arch, seg=t[k], res_ang=res, res_us=res / (360 * 60.0) * 1e6))
    return pd.DataFrame(filas)


def reportes_calibrados(archivos, cal):
    tablas = []
    for a in archivos:
        r = a["reportes"]
        if not len(r):
            continue
        fw = a["fw"]
        kv, ki = _num(fw.get("K_V")), _num(fw.get("K_I"))
        cv, ci = _num(fw.get("CAL_ANG_V", 0)), _num(fw.get("CAL_ANG_I", 0))
        r = r.copy()
        r["archivo"] = a["nombre"]
        r["V_c"] = r.V * kv / cal["K_V"]
        r["I_c"] = r.I * ki / cal["K_I"]
        r["angV_c"] = envolver_g(r.angV + cv - cal["CAL_ANG_V"])
        r["angI_c"] = envolver_g(r.angI + ci - cal["CAL_ANG_I"])
        r["angVI"] = envolver_g(r.angV_c - r.angI_c)
        r["lat_total_ms"] = r.lat_total_us / 1e3
        r["lat_listo_ms"] = r.lat_listo_us / 1e3
        tablas.append(r)
    return pd.concat(tablas, ignore_index=True) if tablas else pd.DataFrame()


# ============================================================ algoritmo (SIMULADO)
def simular_tve_vs_frecuencia(P, desv=np.linspace(-2, 2, 41)):
    fs, n, ppc = 4800.0, 1280, 80
    t = tiempos(n, fs, 7e-6)
    filas = []
    for df_ in desv:
        f = 60 + df_
        peor = 0.0
        for fase in np.linspace(0, 2 * np.pi, 8, endpoint=False):
            v = 1960 + 1000 * np.cos(2 * np.pi * f * t + fase)
            d = dft_upmu(v, v, ppc, fs, 7e-6, 1.0)
            ref = 1000 / np.sqrt(2) * np.exp(1j * fase)
            med = d["V"] * np.exp(1j * d["angV"])
            peor = max(peor, abs(med - ref) / abs(ref) * 100)
        filas.append(dict(desviacion_Hz=df_, TVE_max_pct=peor))
    return pd.DataFrame(filas)


# ============================================================ datos de demostracion (SIMULADOS)
def generar_simulados(carpeta, semilla=11):
    """Escribe archivos con el mismo formato del firmware, con errores conocidos. SON DATOS SIMULADOS."""
    rng = np.random.default_rng(semilla)
    os.makedirs(carpeta, exist_ok=True)
    verdad = dict(K_V=0.001112, K_I=0.2011, fase_V=19.4, fase_I=1.15, ppm=850.0, f0=60.012, V=118.6, R=49.5)
    fsr = 4800 * (1 + verdad["ppm"] * 1e-6)

    def bloque(seg, I_on=True, pps=True):
        t = seg + tiempos(1280, fsr, 7.842e-6) + (0 if pps else rng.uniform(0, 0.9))
        f_t = verdad["f0"] + 0.004 * np.sin(2 * np.pi * t / 40)
        ph = 2 * np.pi * (verdad["f0"] * t - 0.004 * 40 / (2 * np.pi) * (np.cos(2 * np.pi * t / 40) - 1)) + 0.7
        lsb = 3.324 / 4095
        aV = verdad["V"] * np.sqrt(2) * verdad["K_V"] / lsb
        aI = (verdad["V"] / verdad["R"] if I_on else 0) * np.sqrt(2) * verdad["K_I"] / lsb
        forma = lambda p: np.cos(p) + 0.02 * np.cos(3 * p) + 0.01 * np.cos(5 * p)
        v = np.round(1958 + aV * forma(ph + np.radians(verdad["fase_V"])) + rng.normal(0, 1.5, t.size)).astype(int)
        i = np.round(2046 + aI * forma(ph + np.radians(verdad["fase_I"])) + rng.normal(0, 1.5, t.size)).astype(int)
        per = int(round(1e6 * (1 + verdad["ppm"] * 1e-6) + rng.normal(0, 0.6)))
        cab = ["#UPMU2", "#fs=4800", "#n=1280", "#ppc=80", f"#pps={int(pps)}", f"#seg={seg if pps else 0}",
               f"#pps_periodo_us={per if pps else 0}", "#retardo_us=7.042", "#vref=3.3240", "#adcmax=4095", "adcV,adcI"]
        return cab + [f"{a},{b}" for a, b in zip(v, i)] + ["#FIN"]

    def escribir(nombre, meta, lineas):
        with open(os.path.join(carpeta, nombre), "w", encoding="utf-8") as fh:
            fh.write("#meta_version=SIMULADO\n")
            for k, v in meta.items():
                fh.write(f"#meta_{k}={v}\n")
            fh.write("\n".join(lineas) + "\n")

    I_ref = verdad["V"] / verdad["R"]
    l = []
    for s in range(100, 110):
        l += bloque(s)
    escribir("SIM_01_R50_cal_c.csv", dict(prueba="R50_cal", rol="calibracion", ref_V=f"{verdad['V'] * 1.002:.1f}",
                                         ref_I=f"{I_ref * 0.997:.3f}", ref_ang="0", ref_f="", modo="c"), l)
    l = []
    for s in range(200, 260, 2):
        l += bloque(s)
    escribir("SIM_02_R50_val_r.csv", dict(prueba="R50_val", rol="validacion", ref_V=f"{verdad['V'] * 0.998:.1f}",
                                         ref_I=f"{I_ref * 1.004:.3f}", ref_ang="0", ref_f="", modo="r"), l)
    l = []
    for s in range(300, 305):
        l += bloque(s, pps=False)
    escribir("SIM_03_libre_f.csv", dict(prueba="libre", rol="otra", ref_V="", ref_I="", ref_ang="", ref_f="", modo="f"), l)
    per = [int(round(1e6 * (1 + verdad["ppm"] * 1e-6) + rng.normal(0, 0.6))) for _ in range(10)]
    escribir("SIM_04_pps_p.csv", dict(prueba="pps", rol="otra", modo="p"), ["#PRUEBA_PPS (10 s)"] + [f"$P,{p}" for p in per] + ["#FIN_PPS"])
    rep = ["#MEDICION", "#K_V=0.001080", "#K_I=0.196500", "#CAL_ANG_V=0.000", "#CAL_ANG_I=0.000",
           "$H,seg,pps,t_us,V,angV,I,angI,f,rocof,P,Q,FP,lat_listo_us"]
    fprev = None
    for s in range(400, 460):
        lin = bloque(s)
        v = np.array([int(x.split(",")[0]) for x in lin[11:-1]])
        i = np.array([int(x.split(",")[1]) for x in lin[11:-1]])
        d = dft_upmu(v, i, 80, fsr, 7.042e-6, 3.324 / 4095)
        V, I = d["V"] / 0.00108, d["I"] / 0.1965
        aV, aI = np.degrees(d["angV"]), np.degrees(d["angI"])
        fi = np.radians(envolver_g(aV - aI))
        roc = "nan" if fprev is None else f"{d['f'] - fprev:.4f}"
        fprev = d["f"]
        lat = int(266_700 / (1 + verdad["ppm"] * 1e-6) + 2600 + rng.normal(0, 40))
        rep.append(f"$F,{s},1,0,{V:.3f},{aV:.3f},{I:.4f},{aI:.3f},{d['f']:.4f},{roc},{V * I * np.cos(fi):.2f},"
                   f"{V * I * np.sin(fi):.2f},{np.cos(fi):.4f},{lat}")
        rep.append(f"$L,{s},{lat + 9100}")
    rep.append("#FIN_MEDICION")
    escribir("SIM_05_reporte_m.csv", dict(prueba="reporte", rol="otra", modo="m"), rep)
    return verdad

In [ ]:
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)
FIGS = []
def guardar_fig(nombre):
    ruta = os.path.join(CARPETA_RESULTADOS, nombre)
    plt.tight_layout(); plt.savefig(ruta, dpi=150); FIGS.append(ruta); plt.show()
def titulo(t):
    return ("SIMULADO — " + t) if SIMULADO else t
def fmt(x, d=3):
    return "—" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{d}f}"

## 1. Lectura de los archivos

In [ ]:
SIMULADO = False
archivos = leer_carpeta(CARPETA_DATOS) if os.path.isdir(CARPETA_DATOS) else []
if not archivos:
    SIMULADO = True
    CARPETA_DATOS = "datos_SIMULADOS"
    generar_simulados(CARPETA_DATOS)
    archivos = leer_carpeta(CARPETA_DATOS)
    print("*" * 78)
    print("  No hay archivos en 'datos/': se generaron DATOS SIMULADOS en 'datos_SIMULADOS/'.")
    print("  Sirven solo para comprobar el cuaderno. NO SON MEDICIONES.")
    print("*" * 78)

resumen = pd.DataFrame([dict(archivo=a["nombre"], prueba=a["meta"].get("prueba", ""), rol=a["meta"].get("rol", ""),
                             modo=a["meta"].get("modo", ""), bloques=len(a["bloques"]), reportes=len(a["reportes"]),
                             periodos_PPS=len(a["pps"]), ref_V=a["meta"].get("ref_V", ""), ref_I=a["meta"].get("ref_I", ""),
                             ref_ang=a["meta"].get("ref_ang", ""), ref_f=a["meta"].get("ref_f", "")) for a in archivos])
resumen

## 2. Análisis de cada bloque crudo
Para cada bloque de 1280 muestras (16 ciclos):

- **fs real** $= 4800 \cdot \dfrac{T_{PPS}}{10^6}$, con $T_{PPS}$ el periodo del PPS medido por el MCU en µs: corrige el error del reloj del MCU.
- **Algoritmo de la µPMU** (`dft_upmu`): DFT de 16 ciclos, frecuencia por la pendiente de la fase, ángulo referido al PPS.
- **Referencia fuera de línea** (`ajuste_seno`): ajuste sinusoidal de 4 parámetros, sin fuga espectral.

In [ ]:
df = analizar_bloques(archivos, P)
print(f"{len(df)} bloques analizados")
chequeo = df.groupby("archivo").agg(bloques=("bloque", "size"), PPS=("pps", "mean"), centroV=("mediaV", "mean"),
                                    centroI=("mediaI", "mean"), satV=("satV", "sum"), satI=("satI", "sum"),
                                    mV_rms_V=("Vpin", lambda x: 1e3 * x.mean()), mV_rms_I=("Ipin", lambda x: 1e3 * x.mean()),
                                    ppm=("ppm", "mean"), f_Hz=("f", "mean"), THD_V=("thdV", "mean"))
if (df.satV | df.satI).any():
    print("AVISO: hay bloques saturados (columna satV/satI). Esos bloques no sirven para medir.")
chequeo.round(3)

## 3. Calibración (pruebas con rol = calibración)
**Ganancia de tensión**
$$K_V = \frac{V_{pin,rms}}{V_{ref}}$$
$V_{pin,rms}$: tensión rms en PA4 (sin el offset) · $V_{ref}$: tensión de red medida con el multímetro.

**Ganancia de corriente**
$$K_I = \frac{V_{pin,rms}}{I_{ref}}$$

**Desfase de los sensores** (carga resistiva: el ángulo V–I real es $0°$)
$$\Delta\theta_{VI} = (\theta_V - \theta_I)_{medido} - (\theta_V - \theta_I)_{ref}$$
El desfase total se reparte suponiendo que el canal de corriente adelanta lo calculado para la red $R_g$–$C_g$ (1,14°). El resto se atribuye al canal de tensión (ZMPT101B).

In [ ]:
cal = calibrar(df, P)
if not cal["n_bloques"]:
    raise SystemExit("No hay pruebas con rol 'calibracion'. Captura una (carga resistiva + multímetro) antes de seguir.")
ref_V = df[df.rol == "calibracion"].ref_V.mean(); ref_I = df[df.rol == "calibracion"].ref_I.mean()
vp = df[df.rol == "calibracion"].Vpin.mean();      ip = df[df.rol == "calibracion"].Ipin.mean()
print(titulo("CALIBRACIÓN"), "| archivos:", ", ".join(cal["archivos"]))
print(f"\nK_V = V_pin / V_ref = {vp*1e3:.3f} mV / {ref_V:.2f} V = {cal['K_V']:.6f}  (desv. {cal['K_V_sd']:.2e}, {cal['n_V']} bloques)")
print(f"K_I = V_pin / I_ref = {ip*1e3:.3f} mV / {ref_I:.3f} A = {cal['K_I']:.5f}  (desv. {cal['K_I_sd']:.2e}, {cal['n_I']} bloques)")
print(f"Desfase V–I de los sensores = {cal['desfase_VI']:.3f}°  (desv. {cal['desfase_VI_sd']:.3f}°, {cal['n_ang']} bloques)")
print(f"  -> corrección canal I = {cal['CAL_ANG_I']:.3f}°   corrección canal V = {cal['CAL_ANG_V']:.3f}°   (reparto: {cal['fase_origen']})")
print("\nPega esto en src/main.cpp (sección Calibración) para que el modo m reporte valores calibrados:")
print(f"const float K_V       = {cal['K_V']:.6f}f;\nconst float K_I       = {cal['K_I']:.5f}f;")
print(f"const float CAL_ANG_V = {cal['CAL_ANG_V']:.3f}f;\nconst float CAL_ANG_I = {cal['CAL_ANG_I']:.3f}f;")
d = aplicar_cal(df, cal)

### Errores encontrados antes de calibrar
Con las constantes que tenía el firmware (`K_V_FIRMWARE`, `K_I_FIRMWARE`, sin corrección de ángulo).

In [ ]:
c0 = d[np.isfinite(d.ref_V)]
eV0 = (c0.Vpin / P["K_V_FIRMWARE"] / c0.ref_V - 1) * 100
c1 = d[np.isfinite(d.ref_I) & (d.Ipin > 20 * d.lsb)]
eI0 = (c1.Ipin / P["K_I_FIRMWARE"] / c1.ref_I - 1) * 100
c2 = c1[np.isfinite(c1.ref_ang)]
ea0 = envolver_g(c2.angV - c2.angI - c2.ref_ang)
errores_encontrados = pd.DataFrame([
    dict(magnitud="Tensión", error_antes=f"{eV0.mean():+.2f} %", causa="ganancia del ZMPT (potenciómetro) distinta del valor provisional"),
    dict(magnitud="Corriente", error_antes=f"{eI0.mean():+.2f} %", causa="K_I del modelo frente al real (Rb, ganancia del LM358, SCT)"),
    dict(magnitud="Ángulo V–I", error_antes=f"{ea0.mean():+.2f} °", causa="desfase propio de los sensores (sobre todo el módulo ZMPT101B)"),
])
errores_encontrados

In [ ]:
# Forma de onda y diagrama fasorial de un bloque calibrado
fila = d[(d.Ipin > 20 * d.lsb)].iloc[0] if (d.Ipin > 20 * d.lsb).any() else d.iloc[0]
arch = next(a for a in archivos if a["nombre"] == fila.archivo)
b = arch["bloques"][int(fila.bloque)]
lsb = fila.lsb; v = np.array(b["v"]); i = np.array(b["i"])
t_ms = tiempos(len(v), fila.fs_real, 7e-6) * 1e3
fig, ax = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [2, 1]})
ax[0].plot(t_ms, (v - v.mean()) * lsb / cal["K_V"], label="tensión de red (V)")
ax2 = ax[0].twinx(); ax2.plot(t_ms, (i - i.mean()) * lsb / cal["K_I"], color="C1", label="corriente (A)")
ax[0].set_xlabel("tiempo desde el PPS (ms)"); ax[0].set_ylabel("tensión (V)"); ax2.set_ylabel("corriente (A)")
ax[0].set_xlim(0, 50); ax[0].set_title(titulo(f"Forma de onda — {fila.prueba}"))
for x, ang, c, lab in [(fila.V, fila.angV_c, "C0", "V"), (0.6 * fila.V, fila.angI_c, "C1", "I (longitud escalada)")]:
    ax[1].annotate("", xy=(x * np.cos(np.radians(ang)), x * np.sin(np.radians(ang))), xytext=(0, 0),
                   arrowprops=dict(arrowstyle="->", color=c, lw=2))
    ax[1].text(x * np.cos(np.radians(ang)) * 1.08, x * np.sin(np.radians(ang)) * 1.08, lab, color=c)
lim = fila.V * 1.25; ax[1].set_xlim(-lim, lim); ax[1].set_ylim(-lim, lim); ax[1].set_aspect("equal")
ax[1].set_xlabel("parte real (V)"); ax[1].set_ylabel("parte imaginaria (V)"); ax[1].set_title("Fasores en el PPS")
guardar_fig("01_forma_de_onda_y_fasores.png")

## 4. Exactitud: errores y TVE (pruebas con rol = validación)
**Error de magnitud:** $\varepsilon = \dfrac{X_{medido}}{X_{ref}} - 1$  ·  **Error de ángulo V–I:** $\delta = (\theta_V-\theta_I)_{medido} - (\theta_V-\theta_I)_{ref}$

**TVE** (IEEE C37.118.1):
$$TVE = \frac{|\hat X - X|}{|X|} = \left|(1+\varepsilon)\,e^{j\delta} - 1\right|$$

- **TVE de la corriente**: magnitud contra el multímetro y ángulo contra la tensión (carga resistiva). Se mide completo.
- **TVE de la tensión**: la magnitud se mide contra el multímetro. El ángulo absoluto (contra el UTC) no tiene referencia en el banco, así que entra su **incertidumbre**: la suposición sobre la fase del canal I combinada con el retardo PPS→muestra.

El multímetro también tiene error (`EXACT_MULT_*`): no se puede afirmar una exactitud mejor que la suya.

**Cómo bajar la incertidumbre del ángulo absoluto (de ~2° a ~0,2°).** Mide con osciloscopio el desfase de **un** canal y ponlo en `FASE_I_MEDIDA_OSC_DEG` o en `FASE_V_MEDIDA_OSC_DEG`; luego pon `INCERT_FASE_ABS_DEG = 0.2`.
- *Canal de corriente:* usa una fuente de **baja tensión AC aislada** (transformador de 12 V) con una resistencia de potencia. Pinza alrededor de un cable. Canal 1 del osciloscopio sobre la resistencia (su tensión está en fase con la corriente) y canal 2 en **AI** (J4). El desfase de canal 2 respecto a canal 1 es la fase del canal I (+ si adelanta). Usa una corriente parecida a la de la prueba.
- *Canal de tensión:* la misma fuente de 12 V AC en la entrada del ZMPT. Canal 1 en la salida del transformador y canal 2 en **AV** (J4).
Nunca conectes la punta del osciloscopio a la red de 120 V.

In [ ]:
ex = exactitud(d, P)
if ex["aviso"]:
    print("AVISO:", ex["aviso"])
def resumen_serie(s, u):
    return f"media {s.mean():+.3f} {u} | desv. {s.std():.3f} {u} | máx |x| {s.abs().max():.3f} {u}" if len(s) else "sin datos"
print(titulo("EXACTITUD"), f"({ex['n']} bloques de {', '.join(ex['archivos'])})")
print("Error de magnitud V :", resumen_serie(ex["eV"], "%"))
print("Error de magnitud I :", resumen_serie(ex["eI"], "%"))
print("Error de ángulo V–I :", resumen_serie(ex["ea"], "°"))
print("TVE de la corriente :", resumen_serie(ex["tveI"], "%"))
print(f"Incertidumbre del ángulo absoluto de V: ±{ex['u_ang_abs']:.3f}° (= {np.radians(ex['u_ang_abs'])*100:.2f} % de TVE)")
print("TVE de la tensión    :", resumen_serie(ex["tveV"], "%"))
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
ax[0].plot(ex["eV"].values, "o-", label="tensión"); ax[0].plot(ex["eI"].values, "s-", label="corriente")
ax[0].axhline(0, color="k", lw=0.8); ax[0].set_xlabel("bloque"); ax[0].set_ylabel("error de magnitud (%)"); ax[0].legend()
ax[0].set_title(titulo("Error de magnitud"))
ax[1].plot(ex["ea"].values, "o-", color="C2"); ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_xlabel("bloque"); ax[1].set_ylabel("error de ángulo V–I (°)"); ax[1].set_title(titulo("Error de ángulo"))
ax[2].plot(ex["tveI"].values, "o-", label="TVE corriente"); ax[2].plot(ex["tveV"].values, "s-", label="TVE tensión")
ax[2].axhline(CRIT["TVE_PCT"], color="r", ls="--", label="límite 1 %")
ax[2].set_xlabel("bloque"); ax[2].set_ylabel("TVE (%)"); ax[2].legend(); ax[2].set_title(titulo("TVE"))
guardar_fig("02_exactitud_TVE.png")

## 5. Frecuencia y ROCOF
- **FE** $= |f_{medida} - f_{ref}|$: solo se puede calcular si hay una referencia de frecuencia (fuente programable o medidor). Sin referencia se reporta la **dispersión**.
- **ROCOF** entre bloques consecutivos con PPS: $\dfrac{f_k - f_{k-1}}{t_k - t_{k-1}}$, con $t$ en segundos exactos del PPS. En estado estable la red casi no cambia, así que el ROCOF medido es una cota del **RFE**.

In [ ]:
fe_tab = d[np.isfinite(d.ref_f) & d.f_ok & (d.fs_origen != "nominal (sin corregir)")]
FE = (fe_tab.f - fe_tab.ref_f).abs() if len(fe_tab) else pd.Series(dtype=float)
pares = pares_consecutivos(d)
print(titulo("FRECUENCIA"))
fv = d[d.f_ok & (d.fs_origen != "nominal (sin corregir)")]
print(f"f media {fv.f.mean():.4f} Hz | variación entre bloques {fv.f.std()*1e3:.2f} mHz (incluye la variación real de la red)")
if (d.fs_origen == "nominal (sin corregir)").any():
    print("AVISO: hay bloques sin ningún periodo de PPS: su frecuencia no está corregida por el reloj del MCU.")
print(f"Diferencia algoritmo (DFT) vs ajuste sinusoidal: máx {np.max(np.abs(d.f - d.f_aj))*1e3:.2f} mHz")
print("FE contra referencia:", f"máx {FE.max()*1e3:.2f} mHz" if len(FE) else "sin referencia de frecuencia (pon ref_f al capturar)")
if len(pares):
    print(f"ROCOF entre bloques: desv. {pares.rocof.std():.4f} Hz/s | máx |ROCOF| {pares.rocof.abs().max():.4f} Hz/s ({len(pares)} pares)")
rep = reportes_calibrados(archivos, cal)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for arch_, g in d[(d.pps == 1) & d.f_ok].groupby("archivo"):
    if len(g) > 2:
        ax[0].plot(g.seg - g.seg.min(), g.f, ".-", label=f"bloques: {g.prueba.iloc[0]}")
if len(rep):
    for arch_, g in rep[np.isfinite(rep.f)].groupby("archivo"):
        ax[0].plot(g.seg - g.seg.min(), g.f, ".-", label=f"modo m: {arch_[:28]}")
ax[0].set_xlabel("tiempo (s, segundos del PPS)"); ax[0].set_ylabel("frecuencia (Hz)"); ax[0].legend(fontsize=7)
ax[0].set_title(titulo("Frecuencia medida"))
if len(pares):
    ax[1].plot(pares.seg - pares.seg.min(), pares.rocof, "o-")
ax[1].axhline(0, color="k", lw=0.8); ax[1].set_xlabel("tiempo (s)"); ax[1].set_ylabel("ROCOF (Hz/s)")
ax[1].set_title(titulo("ROCOF entre bloques"))
guardar_fig("03_frecuencia_rocof.png")

## 6. Sincronía y reloj del MCU
**Coherencia de fase:** como cada bloque arranca en un PPS, la fase de la tensión en el PPS debe seguir una curva suave en el tiempo (la frecuencia de la red cambia despacio). Para cada bloque se predice su fase con una parábola ajustada a sus vecinos y se compara:
$$\Delta t = \frac{\theta_{medido} - \theta_{predicho}}{360° \cdot f_0}$$
Si la captura no estuviera atada al PPS, los ángulos saltarían al azar (±8333 µs).

**Reloj del MCU:** $\text{ppm} = T_{PPS} - 10^6$ (µs). El firmware lo corrige usando la fs real.

**Retardo PPS→muestra (osciloscopio):** canal 1 en el PPS, canal 2 en PB3 (D13). El retardo esperado es $1/f_s + \text{latencia} \approx 210$ µs. Lo que difiera de lo que supone el firmware va en `CORRECCION_RETARDO_US`.

In [ ]:
coh = coherencia_sincronia(d)
pps_todos = np.concatenate([a["pps"] for a in archivos if len(a["pps"])]) if any(len(a["pps"]) for a in archivos) else np.array([])
ppm_bloques = d.ppm.dropna()
print(titulo("SINCRONÍA"))
if len(coh):
    print(f"Coherencia de fase: rms {np.sqrt(np.mean(coh.res_us**2)):.2f} µs | máx {coh.res_us.abs().max():.2f} µs ({len(coh)} bloques)")
else:
    print("Coherencia de fase: hace falta una ráfaga (modo r) de al menos 7 bloques con PPS")
if len(pps_todos):
    print(f"Prueba PPS: periodo medio {pps_todos.mean():.1f} µs -> reloj del MCU {pps_todos.mean()-1e6:+.1f} ppm | jitter (desv.) {pps_todos.std():.2f} µs")
if len(ppm_bloques):
    print(f"Error del reloj en los bloques: {ppm_bloques.mean():+.1f} ppm (corregido con la fs real)")
libres = d[(d.pps == 0) & d.f_ok]
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
if len(coh):
    for arch_, g in coh.groupby("archivo"):
        ax[0].plot(g.seg - coh.seg.min(), g.res_us, "o-", label=arch_[:30])
    ax[0].axhline(CRIT["SINC_US"], color="r", ls="--", label="±26,5 µs (TVE 1 %)")
    ax[0].axhline(-CRIT["SINC_US"], color="r", ls="--"); ax[0].legend()
ax[0].set_xlabel("tiempo (s)"); ax[0].set_ylabel("error de tiempo equivalente (µs)"); ax[0].set_title(titulo("Coherencia de fase con el PPS"))
sinc = d[(d.pps == 1) & d.f_ok]
ax[1].plot(np.arange(len(sinc)), sinc.angV_aj, "o", label="con PPS")
ax[1].plot(np.arange(len(libres)), libres.angV_aj, "x", label="libres (sin PPS)")
ax[1].set_xlabel("bloque"); ax[1].set_ylabel("ángulo de V en el inicio (°)"); ax[1].legend()
ax[1].set_title(titulo("Ángulo con y sin PPS"))
guardar_fig("04_sincronia.png")

## 7. Reporte de fasores en el MCU y latencia
En el modo **m** el MCU calcula los fasores y manda una línea por segundo. La **latencia** es el tiempo entre el PPS (instante del fasor) y el último byte enviado:
$$\text{latencia} = T_{ventana} + T_{cálculo} + T_{envío}, \quad T_{ventana} = \frac{1280}{4800} = 266{,}7\ \text{ms}$$

In [ ]:
print(titulo("REPORTE Y LATENCIA"))
if len(rep):
    lt = rep.lat_total_ms.dropna(); ll = rep.lat_listo_ms.dropna()
    print(f"{len(rep)} reportes | latencia total media {lt.mean():.1f} ms | máx {lt.max():.1f} ms | "
          f"cálculo ~{(ll.mean() - 1280/4800*1e3):.1f} ms | envío ~{(lt.mean()-ll.mean()):.1f} ms")
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
    for arch_, g in rep.groupby("archivo"):
        ax[0].plot(g.seg - g.seg.min(), g.V_c, ".-", label="V (V)")
        ax0b = ax[0].twinx(); ax0b.plot(g.seg - g.seg.min(), g.I_c, ".-", color="C1"); ax0b.set_ylabel("corriente (A)")
        ax[1].plot(g.seg - g.seg.min(), g.angVI.where(g.I_c > 0.05), ".-", color="C2")
    ax[0].set_xlabel("tiempo (s)"); ax[0].set_ylabel("tensión (V)"); ax[0].set_title(titulo("Reporte: magnitudes"))
    ax[1].set_xlabel("tiempo (s)"); ax[1].set_ylabel("ángulo V–I (°)"); ax[1].set_title(titulo("Reporte: ángulo V–I"))
    lim_ms = CRIT["LATENCIA_FACTOR_P"] / P["REPORTES_POR_S"] * 1e3
    ax[2].hist(lt, bins=20)
    ax[2].set_xlabel("latencia total (ms)"); ax[2].set_ylabel("número de reportes")
    ax[2].set_title(titulo(f"Latencia (límite {lim_ms:.0f} ms)"))
    guardar_fig("05_reporte_latencia.png")
else:
    print("No hay archivos del modo m.")

## 8. Algoritmo
**(a) Simulación (SIMULADO por definición):** TVE de la DFT de 16 ciclos cuando la frecuencia se aleja de 60 Hz, con la corrección de magnitud y de fase que lleva el firmware.

**(b) Datos reales:** diferencia entre el algoritmo de la µPMU y el ajuste sinusoidal sobre los mismos bloques. Mide cuánto error agrega el algoritmo por sí solo (armónicos, fuga y ruido).

In [ ]:
sim = simular_tve_vs_frecuencia(P)
dentro = sim[sim.TVE_max_pct <= CRIT["TVE_PCT"]].desviacion_Hz
print(f"SIMULADO: TVE <= 1 % entre {dentro.min():+.2f} y {dentro.max():+.2f} Hz de desviación")
ok = d[d.Ipin > 20 * d.lsb]
tve_alg = np.abs(ok.Vpin / ok.Vpin_aj * np.exp(1j * np.radians(envolver_g(ok.angV - ok.angV_aj))) - 1) * 100
print(titulo(f"Algoritmo vs ajuste en datos: TVE máx {tve_alg.max():.3f} % | THD tensión {d.thdV.mean():.2f} %"))
plt.figure(figsize=(7, 3.8))
plt.plot(sim.desviacion_Hz, sim.TVE_max_pct, "o-"); plt.axhline(CRIT["TVE_PCT"], color="r", ls="--", label="límite 1 %")
plt.yscale("log"); plt.xlabel("desviación de frecuencia (Hz)"); plt.ylabel("TVE máximo (%)"); plt.legend()
plt.title("SIMULADO — TVE del algoritmo fuera de 60 Hz")
guardar_fig("06_algoritmo_tve_vs_frecuencia.png")

## 9. Resultados finales y criterios de aceptación

In [ ]:
def si_no(c):
    if c == "info":
        return "Informativo"
    return "Sí" if c is True else ("No" if c is False else "No evaluable")
filas = []
def agregar(metrica, valor, criterio, cumple, fuente):
    filas.append(dict(Métrica=metrica, Resultado=valor, Criterio=criterio, Cumple=si_no(cumple), Fuente=fuente))

tI = ex["tveI"].max() if len(ex["tveI"]) else np.nan
tV = ex["tveV"].max() if len(ex["tveV"]) else np.nan
agregar("TVE corriente (máx)", f"{fmt(tI)} %", "≤ 1 %", bool(tI <= CRIT["TVE_PCT"]) if np.isfinite(tI) else None, "IEEE C37.118.1 (estado estable)")
agregar("TVE tensión (máx, con incertidumbre angular)", f"{fmt(tV)} %", "≤ 1 %", bool(tV <= CRIT["TVE_PCT"]) if np.isfinite(tV) else None, "IEEE C37.118.1 (estado estable)")
for nombre, s, lim in [("Error magnitud V |media|+2σ", ex["eV"], CRIT["ERR_V_PCT"]), ("Error magnitud I |media|+2σ", ex["eI"], CRIT["ERR_I_PCT"])]:
    val = abs(s.mean()) + 2 * s.std() if len(s) > 1 else np.nan
    agregar(nombre, f"{fmt(val)} %", f"≤ {lim} %", bool(val <= lim) if np.isfinite(val) else None, "Propio")
ea = ex["ea"]; val = abs(ea.mean()) + 2 * ea.std() if len(ea) > 1 else np.nan
agregar("Error ángulo V–I |media|+2σ", f"{fmt(val)} °", f"≤ {CRIT['ERR_ANG_DEG']} °", bool(val <= CRIT["ERR_ANG_DEG"]) if np.isfinite(val) else None, "Propio (equivale a TVE 1 %)")
fe_alg = np.max(np.abs(d[d.f_ok & (d.fs_origen != "nominal (sin corregir)")].f - d[d.f_ok & (d.fs_origen != "nominal (sin corregir)")].f_aj))
fe_base = 60.0 * 1e-6                     # 1 ppm de resolución del periodo del PPS
agregar("FE estimada: algoritmo (DFT vs ajuste) + base de tiempo", f"{(fe_alg + fe_base)*1e3:.2f} mHz", "≤ 5 mHz",
        bool(fe_alg + fe_base <= CRIT["FE_HZ"]), "IEEE C37.118.1 — sin referencia externa")
if len(FE):
    agregar("FE contra referencia externa (máx)", f"{FE.max()*1e3:.2f} mHz", "≤ 5 mHz", bool(FE.max() <= CRIT["FE_HZ"]), "IEEE C37.118.1")
else:
    agregar("FE contra referencia externa", "sin referencia", "≤ 5 mHz", None, "Captura con ref_f (fuente programable)")
if len(pares):
    rmax = pares.rocof.abs().max()
    agregar("RFE estado estable (máx |ROCOF|)", f"{rmax:.4f} Hz/s", f"≤ {CRIT['RFE_M_HZS']} Hz/s (M) / {CRIT['RFE_P_HZS']} (P)",
            bool(rmax <= CRIT["RFE_M_HZS"]), "IEEE C37.118.1a-2014")
if len(rep):
    lmax = rep.lat_total_ms.max(); lim = CRIT["LATENCIA_FACTOR_P"] / P["REPORTES_POR_S"] * 1e3
    agregar("Latencia del reporte (máx)", f"{lmax:.1f} ms", f"≤ 2/Fs = {lim:.0f} ms (Fs = {P['REPORTES_POR_S']:g}/s)", bool(lmax <= lim), "IEEE C37.118.1 (clase P)")
if len(coh):
    cmax = coh.res_us.abs().max()
    agregar("Sincronía: coherencia de fase (máx)", f"{cmax:.2f} µs", f"≤ {CRIT['SINC_US']} µs", bool(cmax <= CRIT["SINC_US"]), "Equivale a TVE 1 %")
if len(pps_todos):
    agregar("Reloj del MCU (corregido con el PPS)", f"{pps_todos.mean()-1e6:+.1f} ppm", "informativo", "info", "Prueba PPS")
agregar("Rango de frecuencia con TVE ≤ 1 % (simulado)", f"{dentro.min():+.2f} a {dentro.max():+.2f} Hz", "±2 Hz (clase P)",
        bool(dentro.min() <= -2 and dentro.max() >= 2), "IEEE C37.118.1 — SIMULADO")
agregar("Tiempo de respuesta (ventana)", f"{1280/4800*1e3:.1f} ms", f"≤ 1,7/f0 = {1.7/60*1e3:.1f} ms (clase P)", False, "IEEE C37.118.1 — por diseño")

final = pd.DataFrame(filas)
if SIMULADO:
    final.insert(0, "Datos", "SIMULADOS")
final.to_csv(os.path.join(CARPETA_RESULTADOS, "resultados_finales.csv"), index=False, encoding="utf-8-sig")
d.to_csv(os.path.join(CARPETA_RESULTADOS, "bloques_analizados.csv"), index=False, encoding="utf-8-sig")
pd.DataFrame([cal]).drop(columns=["archivos"]).to_csv(os.path.join(CARPETA_RESULTADOS, "calibracion.csv"), index=False, encoding="utf-8-sig")
print(titulo("RESULTADOS FINALES"), "— guardados en", CARPETA_RESULTADOS)
final